# 07 - Train M0: small restoration-only U-Net

**GPU required.** Select the Colab GPU kernel. Run cells in order. This is a first feasibility run, not the final multi-seed experiment. No GAN, intervention map input, minutiae loss or FVC2004 evaluation.

Train on the locked train split, with the exact simulator from notebook 06. References are original impressions, NOT guaranteed clean. Byte-identical train images are used once; images without the simulator's minimum textured foreground are explicitly reported and excluded from training only. Validation images are never filtered by quality.

Model: residual U-Net, base width 16, three downsampling stages, no BatchNorm. Its zero-initialized output layer starts as an identity mapping. Input is a single degraded grayscale channel; output has the same size, clamped to [0,1]. L1 reconstruction loss only. Intervention maps are not supplied to the model.

Eight epochs, batch size 8, 192x192 patches, Adam lr 0.0002. Each train image produces one foreground-centered random patch per epoch. Full-image degradation happens BEFORE cropping. Identity/no-degradation probability is 10%; each of the nine family/severity combinations has probability 10%. Seeds depend on image, epoch and fixed experiment seed; no flips or geometric distortions.

Validation: deterministically select 12 images per source database from validation by seed hash, using all 10 conditions per image. Fixed foreground-centered crops and fixed degradation seeds across epochs. Select the best epoch by mean synthetic validation patch L1. Log identity-image errors separately. This does not measure recognition improvement; notebook 08 must evaluate matching with unchanged galleries and the locked pair lists.

Each completed epoch is copied to Drive, hash checked, then published through latest/best pointer files. After an interruption or error, use Run All to reload the latest completed epoch; do not rerun only the training cell with partially updated weights. An interrupted epoch is repeated. Keep the same runtime for speed; after runtime deletion restore data with notebook 02. Do not edit the frozen experiment configuration to resume. Never load checkpoints from untrusted sources.

Persistent outputs: MyDrive/FingerprintResearch/experiments/m0_unet_v1_seed20261110. The exported training module is reused for later inference. Inspect validation previews and matching before claiming any improvement.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import hashlib
import importlib.util
import json
import math
import random
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('GPU required. Select a Colab GPU runtime before training.')
DEVICE = torch.device('cuda')
SEED = 20261110
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
LOCAL_EXPERIMENT = LOCAL_ROOT / 'training' / 'm0_unet_v1_seed20261110'
for directory in (EXPERIMENT_DIR, LOCAL_EXPERIMENT):
    directory.mkdir(parents=True, exist_ok=True)

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('Experiment:', EXPERIMENT_DIR)

In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
for name, expected in split_protocol['file_sha256'].items():
    if file_digest(SPLIT_DIR / name) != expected:
        raise RuntimeError(f'Frozen split file changed: {name}')
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
normalization = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
simulator_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
if normalization['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json'):
    raise RuntimeError('Normalization references a different split protocol.')
if simulator_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json'):
    raise RuntimeError('Simulator and normalization protocols differ.')
if simulator_config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json'):
    raise RuntimeError('Simulator references a different split protocol.')
if file_digest(SIM_DIR / 'fingerprint_degradation.py') != simulator_config['module_sha256']:
    raise RuntimeError('Frozen simulator source changed.')
if (Image.__version__ != simulator_config['pillow_version']
        or np.__version__ != simulator_config['numpy_version']):
    raise RuntimeError('Pillow/NumPy differ from the frozen simulator environment.')
simulator = load_module('fingerprint_degradation_v1', SIM_DIR / 'fingerprint_degradation.py')
print('Simulator verified:', simulator_config['module_sha256'])

In [ ]:
TRAINING_SOURCE = r'''"""M0 restoration model and deterministic synthetic patch dataset."""
import hashlib
from functools import lru_cache
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset

def stable_seed(text):
    return int.from_bytes(hashlib.sha256(text.encode('utf-8')).digest()[:8], 'big')

@lru_cache(maxsize=64)
def normalized_image(path, dpi):
    with Image.open(path) as image:
        image = image.convert('L')
        if dpi != 500:
            image = image.resize(tuple(max(1, round(length * 500 / dpi)) for length in image.size),
                                 resample=Image.Resampling.LANCZOS)
        result = np.asarray(image, dtype=np.float32) / 255
    result.setflags(write=False)
    return result

class FingerprintPairs(Dataset):
    def __init__(self, records, data_root, simulator, dpi_map, seed, patch_size=192, training=True):
        self.records = list(records)
        if not self.records:
            raise ValueError('Dataset must contain at least one source image.')
        if patch_size < 16:
            raise ValueError('Patch size must be >= 16.')
        self.data_root = Path(data_root)
        self.simulator = simulator
        self.dpi_map = dict(dpi_map)
        self.seed = seed
        self.patch_size = patch_size
        self.training = training
        self.epoch = 0
        self.conditions = [('identity', 'none')] + [
            (family, severity) for family in ('dry_like', 'wet_like', 'generic')
            for severity in ('mild', 'moderate', 'severe')]

    def __len__(self):
        return len(self.records) if self.training else len(self.records) * len(self.conditions)

    def __getitem__(self, index):
        record_index = index if self.training else index // len(self.conditions)
        record = self.records[record_index]
        relative = record['relative_path']
        epoch = self.epoch if self.training else 0
        sample_seed = stable_seed(f'{self.seed}|{epoch}|{relative}|{self.training}')
        rng = np.random.default_rng(sample_seed)
        condition_index = int(rng.integers(len(self.conditions))) if self.training else index % len(self.conditions)
        family, severity = self.conditions[condition_index]
        dpi = self.dpi_map[f"{record['dataset']}_{record['db']}"]
        x = normalized_image(str(self.data_root / relative), dpi)
        foreground = self.simulator.foreground_mask(x)
        if family == 'identity':
            degraded = x.copy()
        else:
            degradation_seed = self.simulator.case_seed(relative, family, severity, self.seed + epoch)
            degraded, _ = self.simulator.degrade(x, family, severity, degradation_seed)
        # Same crop for every validation condition of a source image.
        crop_rng = np.random.default_rng(stable_seed(f'{self.seed}|{epoch}|{relative}|crop'))
        ys, xs = np.where(foreground)
        anchor = int(crop_rng.integers(len(ys)))
        size = self.patch_size
        top = int(np.clip(int(ys[anchor]) - size // 2, 0, max(0, x.shape[0] - size)))
        left = int(np.clip(int(xs[anchor]) - size // 2, 0, max(0, x.shape[1] - size)))
        pad = ((0, max(0, size - x.shape[0])), (0, max(0, size - x.shape[1])))
        if any(value for pair in pad for value in pair):
            target = np.pad(x, pad, mode='reflect')
            degraded = np.pad(degraded, pad, mode='reflect')
        else:
            target = x
        target = np.array(target[top:top + size, left:left + size], dtype=np.float32, copy=True)
        degraded = np.array(degraded[top:top + size, left:left + size], dtype=np.float32, copy=True)
        return {'input': torch.from_numpy(degraded[None]), 'target': torch.from_numpy(target[None]),
                'condition': f'{family}/{severity}', 'relative_path': relative}

class DoubleConv(nn.Module):
    def __init__(self, incoming, outgoing):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(incoming, outgoing, 3, padding=1), nn.ReLU(),
                                    nn.Conv2d(outgoing, outgoing, 3, padding=1), nn.ReLU())

    def forward(self, x):
        return self.layers(x)

class RestorationUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.enc1 = DoubleConv(1, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.center = DoubleConv(base * 4, base * 8)
        self.dec3 = DoubleConv(base * 12, base * 4)
        self.dec2 = DoubleConv(base * 6, base * 2)
        self.dec1 = DoubleConv(base * 3, base)
        self.head = nn.Conv2d(base, 1, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        center = self.center(F.max_pool2d(e3, 2))
        d3 = self.dec3(torch.cat([F.interpolate(center, size=e3.shape[-2:], mode='bilinear', align_corners=False), e3], dim=1))
        d2 = self.dec2(torch.cat([F.interpolate(d3, size=e2.shape[-2:], mode='bilinear', align_corners=False), e2], dim=1))
        d1 = self.dec1(torch.cat([F.interpolate(d2, size=e1.shape[-2:], mode='bilinear', align_corners=False), e1], dim=1))
        return (x + self.head(d1)).clamp(0, 1)
'''

module_bytes = TRAINING_SOURCE.encode('utf-8')
module_sha256 = hashlib.sha256(module_bytes).hexdigest()
local_module = LOCAL_EXPERIMENT / 'restoration_baseline.py'
local_module.write_bytes(module_bytes)
training = load_module('restoration_baseline_v1', local_module)
print('Training module:', module_sha256)

In [ ]:
train_all = locked[locked['split'].eq('train')].sort_values('relative_path')
train_unique = train_all.drop_duplicates('sha256', keep='first')
selection_log = [
    {'relative_path': path, 'decision': 'excluded', 'reason': 'byte_duplicate_train_source'}
    for path in train_all.loc[~train_all.index.isin(train_unique.index), 'relative_path']
]
validation_all = locked[locked['split'].eq('validation')].copy()
validation_all['selection_key'] = validation_all['relative_path'].map(
    lambda path: hashlib.sha256(f'{SEED}|validation_selection|{path}'.encode('utf-8')).hexdigest())
validation_selected = validation_all.sort_values('selection_key').groupby(['dataset', 'db'], sort=True).head(12)
if len(validation_selected) != 72:
    raise RuntimeError('Expected 12 validation sources per source database (72 total).')
train_records = []
for split_name, frame in (('train', train_unique), ('validation', validation_selected)):
    for index, record in enumerate(frame.to_dict('records'), start=1):
        if record['dataset'] not in ('FVC2000', 'FVC2002'):
            raise RuntimeError('Non-development source detected.')
        path = DATA_ROOT / record['relative_path']
        if not path.is_file() or file_digest(path) != record['sha256']:
            raise RuntimeError(f'Missing/changed source: {path}. Restore data with notebook 02.')
        dpi = normalization['source_dpi'][f"{record['dataset']}_{record['db']}"]
        image = training.normalized_image(str(path), dpi)
        try:
            simulator.foreground_mask(image)
        except ValueError as error:
            if split_name != 'train':
                raise RuntimeError(f'Validation source failed foreground check: {path}; do not filter it.') from error
            selection_log.append({'relative_path': record['relative_path'], 'decision': 'excluded', 'reason': str(error)})
            continue
        if split_name == 'train':
            train_records.append(record)
            selection_log.append({'relative_path': record['relative_path'], 'decision': 'included', 'reason': 'passed_source_checks'})
        if index % 250 == 0:
            print(f'Checked {split_name}: {index}/{len(frame)}', flush=True)
train_sources = pd.DataFrame(train_records).sort_values('relative_path').reset_index(drop=True)
validation_sources = validation_selected.drop(columns='selection_key').sort_values('relative_path').reset_index(drop=True)
if len(train_sources) < 128:
    raise RuntimeError('Too few eligible training sources; inspect source-selection log.')
if set(train_sources['identity_key']) & set(validation_sources['identity_key']):
    raise RuntimeError('Train/validation identity overlap.')
if set(train_sources['sha256']) & set(validation_sources['sha256']):
    raise RuntimeError('Train/validation byte overlap.')
selection = pd.DataFrame(selection_log).sort_values('relative_path').reset_index(drop=True)
print('Train sources:', len(train_sources), '| Validation sources:', len(validation_sources))
print(selection.groupby(['decision', 'reason']).size().to_string())

In [ ]:
CONFIG = {
    'experiment': 'm0_unet_v1_seed20261110', 'seed': SEED,
    'epochs': 8, 'batch_size': 8, 'patch_size': 192, 'base_channels': 16,
    'learning_rate': 0.0002, 'optimizer': 'Adam', 'loss': 'L1', 'num_workers': 0,
    'amp': True, 'deterministic_algorithms': True,
    'train_source_policy': 'unique byte hashes; minimum textured foreground; original references not guaranteed clean',
    'validation_policy': '12 hash-selected images per db, all 10 conditions, fixed crops',
    'selection_metric': 'mean synthetic validation patch L1',
    'split_protocol_sha256': file_digest(SPLIT_DIR / 'protocol.json'),
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'simulator_module_sha256': simulator_config['module_sha256'],
    'training_module_sha256': module_sha256,
    'torch_version': str(torch.__version__), 'torch_cuda': str(torch.version.cuda),
    'numpy_version': np.__version__, 'pillow_version': Image.__version__,
}
def csv_bytes(frame):
    return frame.to_csv(index=False, lineterminator='\n').encode('utf-8')

source_files = {
    'train_sources.csv': csv_bytes(train_sources),
    'validation_sources.csv': csv_bytes(validation_sources),
    'source_selection.csv': csv_bytes(selection),
    'restoration_baseline.py': module_bytes,
}
CONFIG['source_file_sha256'] = {name: hashlib.sha256(data).hexdigest() for name, data in source_files.items()}
config_data = json_bytes(CONFIG)
config_sha256 = hashlib.sha256(config_data).hexdigest()
frozen_files = {**source_files, 'configuration.json': config_data}
for name, data in frozen_files.items():
    path = EXPERIMENT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen experiment changed: {path}. Do not overwrite or resume incompatible settings.')
for name, data in frozen_files.items():
    path = EXPERIMENT_DIR / name
    if not path.exists():
        with path.open('xb') as output:
            output.write(data)
    if path.read_bytes() != data:
        raise RuntimeError(f'Experiment persistence verification failed: {path}')
print('Experiment frozen:', config_sha256)

_m0_training_cell_entered = False
train_dataset = training.FingerprintPairs(train_sources.to_dict('records'), DATA_ROOT, simulator,
    normalization['source_dpi'], SEED, patch_size=CONFIG['patch_size'], training=True)
validation_dataset = training.FingerprintPairs(validation_sources.to_dict('records'), DATA_ROOT, simulator,
    normalization['source_dpi'], SEED, patch_size=CONFIG['patch_size'], training=False)
validation_loader = DataLoader(validation_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                               num_workers=0, pin_memory=True)
model = training.RestorationUNet(base=CONFIG['base_channels']).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scaler = torch.amp.GradScaler('cuda', enabled=CONFIG['amp'])
print('Model parameters:', sum(parameter.numel() for parameter in model.parameters()))
print('Train patches/epoch:', len(train_dataset), '| Fixed validation patches:', len(validation_dataset))

In [ ]:
@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    totals = {}
    for batch in loader:
        inputs = batch['input'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)
        # FP32 validation avoids AMP rounding in checkpoint selection.
        predictions = model(inputs).float()
        if not torch.isfinite(predictions).all():
            raise RuntimeError('Non-finite validation predictions.')
        mae = (predictions - targets).abs().mean(dim=(1, 2, 3)).cpu().tolist()
        mse = (predictions - targets).square().mean(dim=(1, 2, 3)).cpu().tolist()
        input_mae = (inputs - targets).abs().mean(dim=(1, 2, 3)).cpu().tolist()
        for condition, error, squared, before in zip(batch['condition'], mae, mse, input_mae):
            values = totals.setdefault(condition, {'samples': 0, 'mae_sum': 0.0, 'mse_sum': 0.0, 'input_mae_sum': 0.0})
            values['samples'] += 1
            values['mae_sum'] += error
            values['mse_sum'] += squared
            values['input_mae_sum'] += before
    count = sum(value['samples'] for value in totals.values())
    if count == 0:
        raise RuntimeError('Validation loader produced no samples.')
    rows = []
    for condition, value in sorted(totals.items()):
        n = value['samples']
        rows.append({'condition': condition, 'samples': n, 'mae': value['mae_sum'] / n,
                     'mse': value['mse_sum'] / n, 'input_mae': value['input_mae_sum'] / n})
    return {
        'mae': sum(value['mae_sum'] for value in totals.values()) / count,
        'mse': sum(value['mse_sum'] for value in totals.values()) / count,
        'input_mae': sum(value['input_mae_sum'] for value in totals.values()) / count,
        'identity_mae': next(row['mae'] for row in rows if row['condition'] == 'identity/none'),
        'conditions': rows,
    }

def publish_json(path, value):
    data = json_bytes(value)
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Pointer write verification failed: {path}')
    pending.replace(path)

def save_epoch(payload, epoch, is_best):
    local = LOCAL_EXPERIMENT / f'epoch_{epoch + 1:03d}.pt'
    torch.save(payload, local)
    expected = file_digest(local)
    name = f'epoch_{epoch + 1:03d}_{expected[:16]}.pt'
    destination = EXPERIMENT_DIR / name
    if destination.exists() and file_digest(destination) != expected:
        raise RuntimeError(f'Checkpoint filename already has different data: {destination}')
    if not destination.exists():
        pending = destination.with_suffix('.pt.part')
        shutil.copyfile(local, pending)
        if file_digest(pending) != expected:
            raise RuntimeError('Checkpoint Drive copy checksum mismatch; latest pointer unchanged.')
        pending.replace(destination)
    pointer = {'file': name, 'sha256': expected, 'epoch': epoch,
               'configuration_sha256': config_sha256, 'validation_mae': payload['history'][-1]['validation_mae']}
    if is_best:
        publish_json(EXPERIMENT_DIR / 'best.json', pointer)
    publish_json(EXPERIMENT_DIR / 'latest.json', pointer)
    local.unlink()

start_epoch = 0
best_mae = float('inf')
history = []
latest_path = EXPERIMENT_DIR / 'latest.json'
if latest_path.exists():
    latest = json.loads(latest_path.read_text(encoding='utf-8'))
    checkpoint_path = EXPERIMENT_DIR / latest['file']
    if latest['configuration_sha256'] != config_sha256 or file_digest(checkpoint_path) != latest['sha256']:
        raise RuntimeError('Latest checkpoint integrity/configuration failure.')
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if checkpoint['configuration_sha256'] != config_sha256:
        raise RuntimeError('Checkpoint payload references a different experiment.')
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scaler.load_state_dict(checkpoint['scaler'])
    torch.set_rng_state(checkpoint['torch_rng'])
    torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
    start_epoch = checkpoint['epoch'] + 1
    best_mae = checkpoint['best_mae']
    history = checkpoint['history']
    print(f'Resuming after completed epoch {start_epoch}; {CONFIG["epochs"] - start_epoch} epochs remain.')
else:
    baseline = evaluate(model, validation_loader, DEVICE)
    if abs(baseline['mae'] - baseline['input_mae']) > 1e-7 or baseline['identity_mae'] > 1e-7:
        raise RuntimeError('Zero-initialized model is not an identity mapping.')
    publish_json(EXPERIMENT_DIR / 'initial_identity_baseline.json', baseline)
    print('Initial input/reference validation MAE:', baseline['input_mae'])

In [ ]:
if _m0_training_cell_entered:
    raise RuntimeError('Use Run All to resume safely; do not rerun only the training cell.')
_m0_training_cell_entered = True
for epoch in range(start_epoch, CONFIG['epochs']):
    started = time.monotonic()
    train_dataset.epoch = epoch
    order = torch.Generator().manual_seed(SEED + epoch)
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                              generator=order, num_workers=0, pin_memory=True)
    model.train()
    loss_sum = 0.0
    seen = 0
    skipped = 0
    for step, batch in enumerate(train_loader, start=1):
        inputs = batch['input'].to(DEVICE, non_blocking=True)
        targets = batch['target'].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=CONFIG['amp']):
            predictions = model(inputs)
            loss = (predictions.float() - targets).abs().mean()
        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite training loss at epoch {epoch + 1}, step {step}.')
        scaler.scale(loss).backward()
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        skipped += int(scaler.get_scale() < previous_scale)
        loss_sum += float(loss.detach()) * len(inputs)
        seen += len(inputs)
        if step % 100 == 0:
            print(f'Epoch {epoch + 1}/{CONFIG["epochs"]}, step {step}/{len(train_loader)}, L1={loss_sum / seen:.5f}', flush=True)
    if seen != len(train_dataset):
        raise RuntimeError('Training epoch did not cover all selected sources.')
    validation_result = evaluate(model, validation_loader, DEVICE)
    is_best = validation_result['mae'] < best_mae
    best_mae = min(best_mae, validation_result['mae'])
    row = {
        'epoch': epoch + 1, 'train_mae': loss_sum / seen,
        'validation_mae': validation_result['mae'], 'validation_mse': validation_result['mse'],
        'validation_input_mae': validation_result['input_mae'],
        'validation_identity_mae': validation_result['identity_mae'],
        'amp_skipped_steps': skipped, 'seconds': time.monotonic() - started,
    }
    history.append(row)
    payload = {
        'configuration_sha256': config_sha256, 'epoch': epoch, 'best_mae': best_mae,
        'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
        'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all(),
        'history': history,
    }
    save_epoch(payload, epoch, is_best)
    pd.DataFrame(history).to_csv(EXPERIMENT_DIR / 'history.csv', index=False)
    pd.DataFrame(validation_result['conditions']).to_csv(EXPERIMENT_DIR / f'validation_epoch_{epoch + 1:03d}.csv', index=False)
    print(f'Epoch {epoch + 1} saved: train={row["train_mae"]:.5f}, val={row["validation_mae"]:.5f}, identity={row["validation_identity_mae"]:.5f}, AMP skips={skipped}, best={is_best}', flush=True)
    start_epoch = epoch + 1

if start_epoch != CONFIG['epochs']:
    raise RuntimeError('Training did not finish the configured number of epochs.')
print('All configured epochs completed and checkpointed.')

In [ ]:
best = json.loads((EXPERIMENT_DIR / 'best.json').read_text(encoding='utf-8'))
best_path = EXPERIMENT_DIR / best['file']
if best['configuration_sha256'] != config_sha256 or file_digest(best_path) != best['sha256']:
    raise RuntimeError('Best checkpoint integrity/configuration failure.')
best_checkpoint = torch.load(best_path, map_location='cpu', weights_only=True)
model.load_state_dict(best_checkpoint['model'])
model.eval()
best_validation = evaluate(model, validation_loader, DEVICE)
publish_json(EXPERIMENT_DIR / 'best_validation.json', best_validation)
pd.DataFrame(best_validation['conditions']).to_csv(EXPERIMENT_DIR / 'best_validation_conditions.csv', index=False)
pd.DataFrame(history).to_csv(EXPERIMENT_DIR / 'history.csv', index=False)
print(pd.DataFrame(best_validation['conditions']).to_string(index=False))
print('Best epoch:', best['epoch'] + 1)
print('Synthetic validation input MAE:', best_validation['input_mae'])
print('Synthetic validation restored MAE:', best_validation['mae'])
print('Identity-image MAE (lower is better):', best_validation['identity_mae'])
if best_validation['mae'] >= best_validation['input_mae']:
    print('WARNING: Best trained model did not beat the input MAE baseline. Do not claim restoration improvement.')

figure, axes = plt.subplots(10, 3, figsize=(9, 24))
with torch.inference_mode():
    for index in range(10):
        sample = validation_dataset[index]
        prediction = model(sample['input'][None].to(DEVICE)).cpu()[0, 0].numpy()
        for column, (name, image) in enumerate((('input', sample['input'][0].numpy()),
                                                ('M0', prediction), ('reference', sample['target'][0].numpy()))):
            axes[index, column].imshow(image, cmap='gray', vmin=0, vmax=1)
            axes[index, column].set_title(f'{sample["condition"]} | {name}', fontsize=8)
            axes[index, column].axis('off')
figure.tight_layout()
figure.savefig(EXPERIMENT_DIR / 'best_validation_preview.png', dpi=140)
plt.show()
completion = {
    'status': 'm0_training_complete', 'configuration_sha256': config_sha256,
    'epochs_completed': len(history), 'best_checkpoint': best,
    'validation_mae': best_validation['mae'], 'input_mae': best_validation['input_mae'],
    'identity_mae': best_validation['identity_mae'],
    'recognition_improvement_verified': False, 'external_test_evaluated': False,
}
publish_json(EXPERIMENT_DIR / 'completion.json', completion)
print('M0 TRAINING COMPLETE:', EXPERIMENT_DIR)
print('Next: full-image inference and matching with unchanged galleries on the locked validation pairs. No real dry/wet or test improvement has been established.')